# LeetCode 3651 — Minimum Cost Path with Teleportations bằng Python

- Link: https://leetcode.com/problems/minimum-cost-path-with-teleportations/
- Độ khó: **Hard**
- Chủ đề: **DP theo lớp** (số lần dịch chuyển), sắp xếp theo giá trị + **min hậu tố**, mô hình đồ thị với **đỉnh ảo**, Dijkstra

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCost(self, grid: List[List[int]], k: int) -> int:
```

## 1. Mục tiêu học

1. Thêm **số lần dịch chuyển đã dùng** `t` vào trạng thái: `f[t][i][j]`.
2. Lớp `t = 0` là DP đường đi phải/xuống kinh điển.
3. Dịch chuyển vào `(x, y)` từ **mọi** ô có giá trị `≥ grid[x][y]` ⇒ min trên một **hậu tố** khi sắp ô theo giá trị — `O(mn log(mn))` mỗi lớp thay vì `O((mn)²)`.
4. Sau khi "đặt" chi phí dịch chuyển, quét lại phải/xuống để đi tiếp trong cùng lớp.
5. Mô hình hoá thành đồ thị có **đỉnh ảo theo giá trị** và chạy Dijkstra — cách nhìn tổng quát.

## 2. Đề bài & ví dụ

Bắt đầu ở `(0, 0)`, tới `(m−1, n−1)`. Di chuyển thường: sang phải hoặc xuống, trả giá trị ô **đích**. Dịch chuyển: từ `(i, j)` tới bất kỳ `(x, y)` với `grid[x][y] ≤ grid[i][j]`, chi phí 0, tối đa `k` lần. Tìm chi phí nhỏ nhất.

**Ví dụ 1:** `grid = [[1,3,3],[2,5,4],[4,3,5]], k = 2` → `7`.

**Ví dụ 2:** `grid = [[1,2],[2,3],[3,4]], k = 1` → `9`.

**Ràng buộc:** `m, n ≤ 80`, `k ≤ 10`, giá trị `≤ 10^4`.

In [1]:
from typing import List
import heapq, random, time

INF = float("inf")
examples = [([[1, 3, 3], [2, 5, 4], [4, 3, 5]], 2, 7), ([[1, 2], [2, 3], [3, 4]], 1, 9)]
for g, k, exp in examples:
    print(g, k, "-> kỳ vọng", exp)

[[1, 3, 3], [2, 5, 4], [4, 3, 5]] 2 -> kỳ vọng 7
[[1, 2], [2, 3], [3, 4]] 1 -> kỳ vọng 9


## 3. Kiến thức nền

### 3.1. DP đường đi phải/xuống

`f[i][j] = grid[i][j] + min(f[i−1][j], f[i][j−1])`, `f[0][0] = 0` (ô xuất phát không tính phí). Duyệt theo hàng rồi cột là thứ tự topo.

In [2]:
def path_dp(grid):
    m, n = len(grid), len(grid[0]); f = [[INF] * n for _ in range(m)]; f[0][0] = 0
    for i in range(m):
        for j in range(n):
            if i: f[i][j] = min(f[i][j], f[i - 1][j] + grid[i][j])
            if j: f[i][j] = min(f[i][j], f[i][j - 1] + grid[i][j])
    return f

print(path_dp([[1, 3, 3], [2, 5, 4], [4, 3, 5]])[-1][-1])

14


### 3.2. Min trên "mọi ô có giá trị ≥ v"

Sắp các ô theo giá trị **giảm dần**; duyệt theo nhóm cùng giá trị, giữ `best` = min của `f[t−1]` trên các ô đã duyệt (giá trị lớn hơn hoặc bằng). Gán `best` cho mọi ô trong nhóm (sau khi đã gộp cả nhóm, vì dịch chuyển giữa các ô **cùng** giá trị là hợp lệ).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

- Lớp `t = 0`: DP 3.1.
- Lớp `t ≥ 1`: trước tiên `f[t][x][y] = min(f[t−1][x][y], min_{grid[i][j] ≥ grid[x][y]} f[t−1][i][j])` (dịch chuyển lần thứ `t` hoặc không dùng thêm), rồi quét phải/xuống: `f[t][i][j] = min(f[t][i][j], f[t][i−1][j] + grid[i][j], f[t][i][j−1] + grid[i][j])`.
- Đáp án `f[k][m−1][n−1]` (đã tính cả "dùng ít hơn `k`" nhờ lấy min với lớp trước).

Ví dụ 1: tới `(1,1)` giá trị 5 tốn `2 + 5 = 7`; dịch chuyển tới `(2,2)` (giá trị 5 ≤ 5) miễn phí ⇒ `7`. ✔

## 5. Cách 1 — Brute Force: Dijkstra với cạnh dịch chuyển đầy đủ

In [3]:
def brute(grid, k):
    m, n = len(grid), len(grid[0]); cells = [(i, j) for i in range(m) for j in range(n)]
    dist = {(0, 0, 0): 0}; pq = [(0, 0, 0, 0)]
    while pq:
        d, i, j, t = heapq.heappop(pq)
        if d > dist[(i, j, t)]: continue
        if (i, j) == (m - 1, n - 1): return d
        nxt = []
        if i + 1 < m: nxt.append((d + grid[i + 1][j], i + 1, j, t))
        if j + 1 < n: nxt.append((d + grid[i][j + 1], i, j + 1, t))
        if t < k:
            for x, y in cells:
                if grid[x][y] <= grid[i][j]: nxt.append((d, x, y, t + 1))
        for nd, x, y, nt in nxt:
            if nd < dist.get((x, y, nt), INF): dist[(x, y, nt)] = nd; heapq.heappush(pq, (nd, x, y, nt))
    return -1

for g, k, exp in examples:
    print(brute(g, k), "kỳ vọng", exp)

7 kỳ vọng 7
9 kỳ vọng 9


**Nút thắt:** mỗi trạng thái có tới `mn` cạnh dịch chuyển ⇒ `O(k · (mn)²)` cạnh — `10 · 6400² ≈ 4·10^8`.

## 6. Ý tưởng thuật toán / bước đột phá

Cạnh dịch chuyển chỉ phụ thuộc **so sánh giá trị** ⇒ gom lại bằng sắp xếp: chi phí tốt nhất để "dịch chuyển tới một ô giá trị `v`" là `min` của lớp trước trên tập ô giá trị `≥ v` — một min hậu tố. Mỗi lớp `O(mn)` (sau khi sắp một lần `O(mn log mn)`), tổng `O(k · mn + mn log mn)`.

## 7. Các cách tiếp theo

- **Cách 2:** DP theo lớp (lời giải nộp).
- **Cách 3:** Dijkstra trên đồ thị lớp với **đỉnh ảo theo giá trị**: ô → đỉnh ảo của giá trị nó (bắt đầu dịch chuyển), đỉnh ảo giá trị `v` → đỉnh ảo giá trị nhỏ hơn kế tiếp (0), đỉnh ảo → các ô có giá trị đó ở lớp kế (0).
- **Cách 4:** DP theo lớp nhưng tính min hậu tố bằng **mảng theo giá trị** (giá trị `≤ 10^4`) thay vì sắp xếp — đếm-sắp.

### 7.1. Cách 2 — DP theo lớp (lời giải chính)

In [4]:
def layered_dp(grid, k):
    m, n = len(grid), len(grid[0])
    cells = sorted(((grid[i][j], i, j) for i in range(m) for j in range(n)), reverse=True)
    f = path_dp(grid)
    for _ in range(k):
        g = [row[:] for row in f]
        best = INF; p = 0
        while p < len(cells):
            q = p; v = cells[p][0]
            while q < len(cells) and cells[q][0] == v:
                best = min(best, f[cells[q][1]][cells[q][2]]); q += 1
            for r in range(p, q):
                _, i, j = cells[r]; g[i][j] = min(g[i][j], best)
            p = q
        for i in range(m):
            for j in range(n):
                if i: g[i][j] = min(g[i][j], g[i - 1][j] + grid[i][j])
                if j: g[i][j] = min(g[i][j], g[i][j - 1] + grid[i][j])
        f = g
    return f[m - 1][n - 1]

for g, k, exp in examples:
    print(layered_dp(g, k), "kỳ vọng", exp)

7 kỳ vọng 7
9 kỳ vọng 9


### 7.2. Cách 3 — Dijkstra với đỉnh ảo theo giá trị

In [5]:
def dijkstra_virtual(grid, k):
    m, n = len(grid), len(grid[0])
    vals = sorted({grid[i][j] for i in range(m) for j in range(n)}, reverse=True)
    vid = {v: r for r, v in enumerate(vals)}; R = len(vals)
    by_val = [[] for _ in range(R)]
    for i in range(m):
        for j in range(n): by_val[vid[grid[i][j]]].append((i, j))
    # đỉnh: ("c", t, i, j) ô ở lớp t; ("v", t, r) đỉnh ảo giá trị vals[r] khi đang dịch chuyển lần t+1
    start = ("c", 0, 0, 0); dist = {start: 0}; pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        if u[0] == "c":
            _, t, i, j = u
            if (i, j) == (m - 1, n - 1): return d
            nxt = []
            if i + 1 < m: nxt.append((d + grid[i + 1][j], ("c", t, i + 1, j)))
            if j + 1 < n: nxt.append((d + grid[i][j + 1], ("c", t, i, j + 1)))
            if t < k: nxt.append((d, ("v", t, vid[grid[i][j]])))
        else:
            _, t, r = u
            nxt = [(d, ("c", t + 1, i, j)) for i, j in by_val[r]]
            if r + 1 < R: nxt.append((d, ("v", t, r + 1)))
        for nd, w in nxt:
            if nd < dist.get(w, INF): dist[w] = nd; heapq.heappush(pq, (nd, w))
    return -1

for g, k, exp in examples:
    print(dijkstra_virtual(g, k), "kỳ vọng", exp)

7 kỳ vọng 7
9 kỳ vọng 9


**Ý nghĩa:** chuỗi đỉnh ảo giảm dần theo giá trị thay cho `O(mn)` cạnh mỗi ô — kỹ thuật "nén cạnh bằng đỉnh trung gian" rất hay dùng khi cạnh được xác định bởi một **thứ tự**.

### 7.3. Cách 4 — Min hậu tố theo mảng giá trị

In [6]:
def layered_counting(grid, k):
    m, n = len(grid), len(grid[0]); V = max(max(r) for r in grid)
    f = path_dp(grid)
    for _ in range(k):
        best_at = [INF] * (V + 2)
        for i in range(m):
            for j in range(n):
                if f[i][j] < best_at[grid[i][j]]: best_at[grid[i][j]] = f[i][j]
        for v in range(V - 1, -1, -1):                       # min hậu tố theo giá trị
            if best_at[v + 1] < best_at[v]: best_at[v] = best_at[v + 1]
        g = [[min(f[i][j], best_at[grid[i][j]]) for j in range(n)] for i in range(m)]
        for i in range(m):
            for j in range(n):
                if i: g[i][j] = min(g[i][j], g[i - 1][j] + grid[i][j])
                if j: g[i][j] = min(g[i][j], g[i][j - 1] + grid[i][j])
        f = g
    return f[m - 1][n - 1]

for g, k, exp in examples:
    print(layered_counting(g, k), "kỳ vọng", exp)

7 kỳ vọng 7
9 kỳ vọng 9


## 8. TRACE từng bước

In [7]:
def trace(grid, k):
    m, n = len(grid), len(grid[0]); V = max(max(r) for r in grid)
    f = path_dp(grid); show = lambda F: [[("∞" if x == INF else x) for x in row] for row in F]
    print("lớp 0:", show(f))
    for t in range(1, k + 1):
        best_at = [INF] * (V + 2)
        for i in range(m):
            for j in range(n): best_at[grid[i][j]] = min(best_at[grid[i][j]], f[i][j])
        for v in range(V - 1, -1, -1): best_at[v] = min(best_at[v], best_at[v + 1])
        g = [[min(f[i][j], best_at[grid[i][j]]) for j in range(n)] for i in range(m)]
        print(f"lớp {t} sau dịch chuyển:", show(g))
        for i in range(m):
            for j in range(n):
                if i: g[i][j] = min(g[i][j], g[i - 1][j] + grid[i][j])
                if j: g[i][j] = min(g[i][j], g[i][j - 1] + grid[i][j])
        print(f"lớp {t} sau quét phải/xuống:", show(g)); f = g
    print("đáp án:", f[-1][-1])

trace([[1, 3, 3], [2, 5, 4], [4, 3, 5]], 2)

lớp 0: [[0, 3, 6], [2, 7, 10], [6, 9, 14]]
lớp 1 sau dịch chuyển: [[0, 3, 3], [2, 7, 6], [6, 3, 7]]
lớp 1 sau quét phải/xuống: [[0, 3, 3], [2, 7, 6], [6, 3, 7]]
lớp 2 sau dịch chuyển: [[0, 3, 3], [2, 7, 6], [6, 3, 7]]
lớp 2 sau quét phải/xuống: [[0, 3, 3], [2, 7, 6], [6, 3, 7]]
đáp án: 7


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def minCost(self, grid: List[List[int]], k: int) -> int:
        m, n = len(grid), len(grid[0])
        INF = float('inf')
        f = [[INF] * n for _ in range(m)]
        f[0][0] = 0
        for i in range(m):
            for j in range(n):
                if i:
                    f[i][j] = min(f[i][j], f[i - 1][j] + grid[i][j])
                if j:
                    f[i][j] = min(f[i][j], f[i][j - 1] + grid[i][j])
        V = max(max(row) for row in grid)
        for _ in range(k):
            best_at = [INF] * (V + 2)
            for i in range(m):
                for j in range(n):
                    v = grid[i][j]
                    if f[i][j] < best_at[v]:
                        best_at[v] = f[i][j]
            for v in range(V - 1, -1, -1):
                if best_at[v + 1] < best_at[v]:
                    best_at[v] = best_at[v + 1]
            g = [[min(f[i][j], best_at[grid[i][j]]) for j in range(n)] for i in range(m)]
            for i in range(m):
                gi = g[i]
                for j in range(n):
                    if i and g[i - 1][j] + grid[i][j] < gi[j]:
                        gi[j] = g[i - 1][j] + grid[i][j]
                    if j and gi[j - 1] + grid[i][j] < gi[j]:
                        gi[j] = gi[j - 1] + grid[i][j]
            f = g
        return f[m - 1][n - 1]

sol = Solution()
for g, k, exp in examples:
    got = sol.minCost(g, k)
    print(got, "✔" if got == exp else "✘")

7 ✔
9 ✔


## 10. Vì sao thuật toán đúng

**Bất biến:** `f_t[i][j]` = chi phí nhỏ nhất tới `(i, j)` dùng **tối đa** `t` lần dịch chuyển.

1. `t = 0`: chỉ đi phải/xuống ⇒ DP 3.1 đúng.
2. Bước `t − 1 → t`: một đường đi tối ưu dùng `≤ t` lần, xét lần dịch chuyển **cuối cùng** (nếu có): trước đó là đường đi dùng `≤ t − 1` lần tới ô `(i, j)`, dịch chuyển tới `(x, y)` với `grid[x][y] ≤ grid[i][j]`, sau đó chỉ đi phải/xuống. Bước "gán min hậu tố" bao mọi lựa chọn `(i, j)`; bước quét phải/xuống bao phần đi sau. Nếu không dùng lần thứ `t`, giá trị bằng `f_{t−1}` (đã lấy min).
3. Chi phí không âm ⇒ dịch chuyển nhiều lần trong cùng một "vị trí" không cần xét riêng (đã bao trong các lớp).
4. Đáp án `f_k[m−1][n−1]`.

## 11. Phân tích Time / Space Complexity

Với `N = mn`, `V = max giá trị`:

| Cách | Time | Space |
|---|---|---|
| 1. Dijkstra cạnh đầy đủ | `O(k N² log)` | `O(k N)` |
| 2. DP lớp + sắp xếp | `O(N log N + k N)` | `O(N)` |
| 3. Dijkstra đỉnh ảo | `O(k (N + V) log)` | `O(k (N + V))` |
| 4. DP lớp + mảng giá trị | `O(k (N + V))` | `O(N + V)` |

## 12. Bộ test

In [9]:
METHODS = [brute, layered_dp, dijkstra_virtual, layered_counting, lambda g, k: Solution().minCost(g, k)]
for g, k, exp in examples:
    assert all(f(g, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [10]:
# 12.2 ca biên
assert Solution().minCost([[5, 1], [1, 9]], 0) == 10                     # không dịch chuyển
assert Solution().minCost([[9, 1], [1, 2]], 1) == 0                      # dịch chuyển thẳng tới đích
assert Solution().minCost([[0, 0], [0, 0]], 3) == 0
print("Ca biên: OK")

Ca biên: OK


In [11]:
# 12.3 ngẫu nhiên
rng = random.Random(3651)
for _ in range(500):
    m = rng.randint(2, 4); n = rng.randint(2, 4); k = rng.randint(0, 3)
    g = [[rng.randint(0, 9) for _ in range(n)] for _ in range(m)]
    b = brute(g, k)
    assert all(f(g, k) == b for f in METHODS[1:]), (g, k)
print("500 test ngẫu nhiên: OK")

500 test ngẫu nhiên: OK


In [12]:
# 12.4 test lớn 80 x 80, k = 10
rng = random.Random(1)
g = [[rng.randint(0, 10**4) for _ in range(80)] for _ in range(80)]
t0 = time.time(); x = Solution().minCost(g, 10); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 2), "s | khớp DP sắp xếp:", x == layered_dp(g, 10))

đáp án: 3797 | thời gian: 0.03 s | khớp DP sắp xếp: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dịch chuyển chỉ tới ô có giá trị **nhỏ hơn hẳn** (bỏ qua bằng nhau)

In [13]:
def wrong_strict(grid, k):
    m, n = len(grid), len(grid[0]); f = path_dp(grid)
    for _ in range(k):
        g = [[min(f[i][j], min((f[x][y] for x in range(m) for y in range(n) if grid[x][y] > grid[i][j]), default=INF))
              for j in range(n)] for i in range(m)]                           # SAI: > thay vì >=
        for i in range(m):
            for j in range(n):
                if i: g[i][j] = min(g[i][j], g[i - 1][j] + grid[i][j])
                if j: g[i][j] = min(g[i][j], g[i][j - 1] + grid[i][j])
        f = g
    return f[-1][-1]

print("sai:", wrong_strict([[1, 3, 3], [2, 5, 4], [4, 3, 5]], 2), "| đúng:", brute([[1, 3, 3], [2, 5, 4], [4, 3, 5]], 2))

sai: 11 | đúng: 7


### Lỗi 2 — Quên quét phải/xuống sau khi dịch chuyển

In [14]:
def wrong_no_resweep(grid, k):
    m, n = len(grid), len(grid[0]); f = path_dp(grid)
    for _ in range(k):
        f = [[min(f[i][j], min((f[x][y] for x in range(m) for y in range(n) if grid[x][y] >= grid[i][j]), default=INF))
              for j in range(n)] for i in range(m)]                           # SAI: không đi tiếp
    return f[-1][-1]

rng = random.Random(5)
while True:
    g = [[rng.randint(0, 9) for _ in range(3)] for _ in range(3)]; kk = rng.randint(1, 2)
    if wrong_no_resweep(g, kk) != brute(g, kk): break
print("phản ví dụ:", g, kk, "| sai:", wrong_no_resweep(g, kk), "| đúng:", brute(g, kk))

phản ví dụ: [[6, 4, 2], [6, 2, 1], [2, 9, 9]] 2 | sai: 15 | đúng: 9


### Lỗi 3 — Tính phí ô xuất phát

In [15]:
def wrong_start_cost(grid, k):
    return Solution().minCost(grid, k) + grid[0][0]                           # SAI

print("sai:", wrong_start_cost([[1, 2], [2, 3], [3, 4]], 1), "| đúng:", brute([[1, 2], [2, 3], [3, 4]], 1))

sai: 10 | đúng: 9


### Lỗi 4 — Gán min hậu tố **trước** khi gộp đủ cả nhóm cùng giá trị

Nếu duyệt từng ô theo giá trị giảm dần và gán ngay `best` hiện tại, các ô cùng giá trị đứng trước trong nhóm sẽ không thấy ô cùng giá trị đứng sau — sai vì dịch chuyển giữa các ô bằng giá trị là hợp lệ.

In [16]:
def wrong_group(grid, k):
    m, n = len(grid), len(grid[0])
    cells = sorted(((grid[i][j], i, j) for i in range(m) for j in range(n)), reverse=True); f = path_dp(grid)
    for _ in range(k):
        g = [row[:] for row in f]; best = INF
        for v, i, j in cells:
            g[i][j] = min(g[i][j], best); best = min(best, f[i][j])           # SAI: gán trước khi gộp
        for i in range(m):
            for j in range(n):
                if i: g[i][j] = min(g[i][j], g[i - 1][j] + grid[i][j])
                if j: g[i][j] = min(g[i][j], g[i][j - 1] + grid[i][j])
        f = g
    return f[-1][-1]

rng = random.Random(6)
while True:
    g = [[rng.randint(0, 3) for _ in range(3)] for _ in range(3)]; kk = rng.randint(1, 2)
    if wrong_group(g, kk) != brute(g, kk): break
print("phản ví dụ:", g, kk, "| sai:", wrong_group(g, kk), "| đúng:", brute(g, kk))

phản ví dụ: [[0, 3, 2], [0, 0, 1], [3, 2, 2]] 1 | sai: 3 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa `k` lần dùng một đặc quyền" ⇒ thêm chiều `t` vào DP, mỗi lớp xây từ lớp trước.
- "Nhảy tới mọi ô có giá trị ≤ v" ⇒ sắp theo giá trị + min hậu tố, hoặc chuỗi **đỉnh ảo** trong đồ thị.
- Trong lớp chỉ có cạnh phải/xuống ⇒ một lần quét theo hàng là đủ (thứ tự topo).

## 15. Biến thể của bài

1. **Di chuyển bốn hướng**: trong mỗi lớp cần Dijkstra thay cho một lần quét.
2. **Dịch chuyển có phí `c`**: cộng `c` khi gán min hậu tố.
3. **Dịch chuyển tới ô giá trị ≥** (ngược chiều): min **tiền tố** theo giá trị.
4. **Không giới hạn `k`**: lặp tới khi lớp ổn định (tối đa số giá trị khác nhau lần).

In [17]:
# Biến thể 2: dịch chuyển tốn phí c
def min_cost_with_fee(grid, k, c):
    m, n = len(grid), len(grid[0]); V = max(max(r) for r in grid); f = path_dp(grid)
    for _ in range(k):
        best_at = [INF] * (V + 2)
        for i in range(m):
            for j in range(n): best_at[grid[i][j]] = min(best_at[grid[i][j]], f[i][j])
        for v in range(V - 1, -1, -1): best_at[v] = min(best_at[v], best_at[v + 1])
        g = [[min(f[i][j], best_at[grid[i][j]] + c) for j in range(n)] for i in range(m)]
        for i in range(m):
            for j in range(n):
                if i: g[i][j] = min(g[i][j], g[i - 1][j] + grid[i][j])
                if j: g[i][j] = min(g[i][j], g[i][j - 1] + grid[i][j])
        f = g
    return f[-1][-1]

print("phí 0:", min_cost_with_fee([[1, 3, 3], [2, 5, 4], [4, 3, 5]], 2, 0), "| phí 3:", min_cost_with_fee([[1, 3, 3], [2, 5, 4], [4, 3, 5]], 2, 3))
assert min_cost_with_fee([[1, 3, 3], [2, 5, 4], [4, 3, 5]], 2, 0) == 7

phí 0: 7 | phí 3: 10


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Dijkstra đầy đủ | Định nghĩa | `O(k N²)` | Kiểm chứng |
| 2. DP lớp + sắp | Min hậu tố theo thứ tự | `O(N log N + kN)` | Giá trị lớn |
| 3. Dijkstra đỉnh ảo | Nén cạnh | `O(k (N+V) log)` | Khi di chuyển tuỳ ý (4 hướng) |
| 4. DP lớp + mảng giá trị | Đếm-sắp | `O(k (N + V))` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `grid = [[3,1],[1,1]], k = 1`.

**Bài 2 (tính tay).** Lớp 0 và lớp 1 cho `grid = [[2,8],[9,1]]`, `k = 1`.

**Bài 3 (điền TODO).** Viết `min_cost_4dir(grid, k)` (di chuyển bốn hướng).

**Bài 4 (tự viết lại).** Trả về **số lần dịch chuyển ít nhất** cần để đạt chi phí tối ưu với `k` cho trước.

**Bài 5 (mini challenge).** Với mọi `t = 0..k`, in chi phí tối ưu khi dùng tối đa `t` lần — và giải thích vì sao dãy không tăng.

In [18]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_cost_4dir(grid, k):
    # TODO: Dijkstra trong từng lớp
    pass

In [19]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def costs_by_t(grid, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không dịch chuyển: `min(1 + 1, 1 + 1) = 2`. (2) Dịch chuyển từ `(0,0)` (giá trị 3) tới đích (giá trị 1): 0. (3) `0`.

**Bài 2.** (1) Lớp 0: `[[0, 8], [9, 9]]`. (2) Lớp 1: dịch chuyển từ `(0,0)` giá trị 2 tới ô giá trị ≤ 2: `(1,1)` ⇒ 0. (3) Đáp án `0`.

**Bài 3.** (1) Mỗi lớp: khởi tạo bằng min hậu tố như cũ. (2) Chạy Dijkstra nhiều nguồn trên lưới (cạnh bốn hướng, phí ô đích). (3) Lấy lớp `k`.

**Bài 4.** (1) Tính `cost_t` cho mọi `t`. (2) Tìm `t` nhỏ nhất có `cost_t = cost_k`. (3) Trả `t`.

**Bài 5.** (1) Lưu đáp án mỗi lớp. (2) Lớp sau lấy `min` với lớp trước. (3) Dãy không tăng vì `f_t ≤ f_{t−1}` theo từng ô.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
assert Solution().minCost([[3, 1], [1, 1]], 1) == 0
assert path_dp([[2, 8], [9, 1]]) == [[0, 8], [9, 9]] and Solution().minCost([[2, 8], [9, 1]], 1) == 0
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [21]:
# Bài 3: bốn hướng — Dijkstra mỗi lớp, so với Dijkstra trạng thái đầy đủ
def min_cost_4dir_sol(grid, k):
    m, n = len(grid), len(grid[0]); V = max(max(r) for r in grid)
    def dijkstra(init):
        d = [row[:] for row in init]; pq = [(d[i][j], i, j) for i in range(m) for j in range(n) if d[i][j] < INF]
        heapq.heapify(pq)
        while pq:
            c, i, j = heapq.heappop(pq)
            if c > d[i][j]: continue
            for x, y in ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1)):
                if 0 <= x < m and 0 <= y < n and c + grid[x][y] < d[x][y]:
                    d[x][y] = c + grid[x][y]; heapq.heappush(pq, (d[x][y], x, y))
        return d
    init = [[INF] * n for _ in range(m)]; init[0][0] = 0; f = dijkstra(init)
    for _ in range(k):
        best_at = [INF] * (V + 2)
        for i in range(m):
            for j in range(n): best_at[grid[i][j]] = min(best_at[grid[i][j]], f[i][j])
        for v in range(V - 1, -1, -1): best_at[v] = min(best_at[v], best_at[v + 1])
        f = dijkstra([[min(f[i][j], best_at[grid[i][j]]) for j in range(n)] for i in range(m)])
    return f[-1][-1]

def brute_4dir(grid, k):
    m, n = len(grid), len(grid[0]); dist = {(0, 0, 0): 0}; pq = [(0, 0, 0, 0)]
    while pq:
        d, i, j, t = heapq.heappop(pq)
        if d > dist[(i, j, t)]: continue
        if (i, j) == (m - 1, n - 1): return d
        nxt = [(d + grid[x][y], x, y, t) for x, y in ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1)) if 0 <= x < m and 0 <= y < n]
        if t < k: nxt += [(d, x, y, t + 1) for x in range(m) for y in range(n) if grid[x][y] <= grid[i][j]]
        for nd, x, y, nt in nxt:
            if nd < dist.get((x, y, nt), INF): dist[(x, y, nt)] = nd; heapq.heappush(pq, (nd, x, y, nt))

rng = random.Random(4)
for _ in range(200):
    m = rng.randint(2, 4); n = rng.randint(2, 4); k = rng.randint(0, 2)
    g = [[rng.randint(0, 9) for _ in range(n)] for _ in range(m)]
    assert min_cost_4dir_sol(g, k) == brute_4dir(g, k)
print("Bài 3 OK")

Bài 3 OK


In [22]:
# Bài 4, 5
def costs_by_t_sol(grid, k):
    return [Solution().minCost(grid, t) for t in range(k + 1)]

def min_teleports_needed(grid, k):
    cs = costs_by_t_sol(grid, k); return cs.index(cs[-1])

rng = random.Random(5)
for _ in range(100):
    g = [[rng.randint(0, 20) for _ in range(5)] for _ in range(5)]; cs = costs_by_t_sol(g, 4)
    assert all(cs[i] >= cs[i + 1] for i in range(len(cs) - 1))
print("Bài 4, 5 OK; ví dụ 1:", costs_by_t_sol(examples[0][0], 3), "| cần tối thiểu", min_teleports_needed(examples[0][0], 3), "lần")

Bài 4, 5 OK; ví dụ 1: [14, 7, 7, 7] | cần tối thiểu 1 lần


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f = DP phải/xuống (f[0][0] = 0)
repeat k lần:
    best_at[v] = min f trên ô giá trị v; min hậu tố theo v giảm dần
    g[i][j] = min(f[i][j], best_at[grid[i][j]])
    quét phải/xuống trên g; f = g
return f[m-1][n-1]
```

**Checklist:**

- [ ] Dịch chuyển tới ô có giá trị **≤** (gồm bằng)?
- [ ] Gộp đủ nhóm cùng giá trị trước khi gán?
- [ ] Quét lại phải/xuống sau mỗi lớp?
- [ ] Ô xuất phát không tính phí?
- [ ] Đã so với Dijkstra trạng thái đầy đủ?